<img src="https://datascientest.fr/train/assets/logo_datascientest.png" style="height:100px" alt="MCUNet Logo">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> Déploiement TinyML avec MCUNet </h1></center>
<center><h2> Notebook 05 : Et si ça ne rentre pas ? (Solution) </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

>
> **Objectif :** Comprendre et arbitrer entre les différentes stratégies d'optimisation quand un modèle dépasse le budget mémoire du MCU.
>
> **Durée estimée :** 30 minutes
>
> **Prérequis :** Avoir fait l'analyse de déploiement (Notebook 04).
>


### Le problème initial

Rappelons la situation du notebook précédent : nous voulons absolument la haute précision de `mcunet-vww2` (91.8%), mais il requiert plus de 310 KB de SRAM, alors que notre STM32F746 n'a que ~270 KB de SRAM disponible pour le modèle ML.

Quelles sont nos options en tant qu'ingénieur Edge AI ?

--- 
### Stratégie 1 : Downgrade du modèle (Le compromis logiciel)

La méthode la plus simple : choisir un modèle plus petit dans le Model Zoo. Nous avons vu que `mcunet-vww1` rentre parfaitement (162 KB).

**Impact :**
- Baisse de la précision Top-1 de 91.8% à 88.9% (Perte de 2.9%).
- Temps de développement : 0 (déjà disponible).
- Coût matériel : 0.

--- 
### Stratégie 2 : Upgrade Hardware (La force brute)

Si l'application métier exige 91.8% de précision (par exemple, des fausses alertes coûtent très cher), nous pouvons changer de microcontrôleur.

Passons sur la gamme au-dessus, le **STM32H743** (Architecture similaire, mais 512 KB de SRAM et 2 MB de Flash).

In [ ]:
import os
import sys
sys.path.append('/workspace/scripts/')
from tinyengine_analysis import analyze_tflite, print_report

tflite_vww2 = os.path.expanduser("~/.mcunet/mcunet-vww2.tflite")

print("Analyse pour STM32H743...")
result_vww2 = analyze_tflite(tflite_vww2)
print_report(result_vww2, "STM32H743")

**Impact :**
- Maintien de la précision (91.8%).
- Temps de développement : Bas (Changement de cible de compilation).
- **Coût matériel :** Le STM32H743 coûte plus cher que le F746 (ex: 8$ vs 6$ l'unité). Sur un produit fabriqué à 1 million d'exemplaires, cette décision coûte **2 millions de dollars** supplémentaires à l'entreprise.

--- 
### Stratégie 3 : Réduction de la résolution d'entrée (petit hack ingénieux)

Si on regarde pourquoi `mcunet-vww2` consomme tant de SRAM, c'est principalement dû à sa résolution d'entrée : 144x144 pixels, contre 80x80 pour `mcunet-vww1`.

La taille des activations au début du réseau évolue quadratiquement avec la résolution. Si on réduit la résolution de 144 à 96, on divise l'aire de l'image par 2.25.

### Petit exercice pratique

Dans l'objet d'analyse, l'estimation de la SRAM est une valeur en KB. Si on estime que le pic mémoire est linéairement proportionnel au nombre de pixels de l'image d'entrée, calculer le nouveau pic SRAM si on passe l'entrée de 144x144 à 96x96.

In [ ]:
original_resolution = 144
new_resolution = 96
original_sram_kb = result_vww2['sram_peak_kb'] # Environ 320-330 KB avec notre estimateur

# Solution
ratio = (new_resolution**2) / (original_resolution**2)
new_sram_kb = original_sram_kb * ratio

print(f"Résolution originale : {original_resolution}x{original_resolution}")
print(f"Nouvelle résolution  : {new_resolution}x{new_resolution}")
print(f"Ratio des aires      : {ratio:.2f}")
print(f"Pic SRAM original    : {original_sram_kb:.1f} KB")
print(f"Nouveau pic estimé   : {new_sram_kb:.1f} KB")

if new_sram_kb < 270:
    print("\nÇa rentre sur le STM32F746")
else:
    print("\nToujours trop grand.")

**Impact :**

- Cela nécessite de **réentraîner** le modèle (ou de faire du fine-tuning) sur la nouvelle résolution, car le réseau est habitué à voir des caractéristiques de taille 144x144.
- Baisse probable de la précision (située quelque part entre vww1 et vww2).
- Temps de développement : Moyen/Haut (Besoin d'un cycle de MLOps complet).

--- 
### Stratégie 4 : Patch-Based Inference (MCUNetV2)

C'est la solution avancée trouvée par les chercheurs du MIT. Au lieu d'inférer toute l'image d'un coup (ce qui crée un énorme tenseur en mémoire sur les premières couches), le moteur d'inférence (TinyEngine) découpe l'image en "patchs" (ex: 4 morceaux) et exécute les premières couches sur un patch à la fois.

Les features maps résultantes de chaque patch (qui sont beaucoup plus petites car le réseau a réduit la résolution spatiale entre-temps) sont ensuite recombinées avant les couches fully connected.

Cette méthode divise le pic SRAM initial par 4 sans perdre aucune information, mais ajoute un overhead de calcul (~10% de MACs en plus pour gérer les bordures des patchs).

--- 
### Questions de réflexion

1. Dans un projet réel en entreprise, votre modèle est à 30 KB au-dessus du budget SRAM. Comment choisiriez-vous entre la stratégie 1 (modèle plus petit) et la stratégie 2 (hardware plus puissant) ? Quels facteurs devez-vous prendre en compte ?

--> C'est un calcul de ROI (Return on Investment). On doit chiffrer le coût financier de l'upgrade matériel (surcam coût unitaire * volume de production) par rapport au coût du modèle moins précis (est-ce que perdre 2% de précision génère des faux positifs qui coûtent cher en SAV, ou dégrade l'expérience utilisateur de manière inacceptable ?). Souvent sur de très gros volumes, on fera tout pour rester sur le matériel le moins cher.

2. La technique des patchs (MCUNetV2) échange du calcul (plus de MACs) contre de la mémoire (moins de SRAM). Pourquoi cet échange est-il généralement gagnant sur un microcontrôleur ?

--> Le microcontrôleur n'a physiquement pas plus de SRAM. Si ça dépasse, le programme crashe (HardFault) ou ne compile pas. On ne peut pas "rajouter" de la SRAM par magie (pas de swap disk). En revanche, le calcul demande juste du temps. Augmenter le temps d'inférence de 10% (ex: passer de 200ms à 220ms) est souvent tout à fait acceptable pour l'application finale, là où manquer de mémoire est fatal.